In [1]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F


In [2]:
#MHA
#首先实现self_attention

class SingleHeadAttention(nn.Module):
    def __init__(self,config):
        super().__init__()
        self.n_embed = config.n_embed
        self.head_size = config.head_size # head_size = n_embed // n_head
        self.q_proj = nn.Linear(config.n_embed,config.head_size)
        self.k_proj = nn.Linear(config.n_embed,config.head_size)
        self.v_proj = nn.Linear(config.n_embed,config.head_size)
        
        self.register_buffer(
            'attention_mask',
            torch.tril(
                torch.ones(config.max_len,config.max_len)
            )
        )
        self.dropout = nn.Dropout(config.dropout)
    
    def forward(self,x):
        batch_size,seq_len,hidden_dim = x.size()
        query = self.q_proj(x) #(B,S,head_size)
        key = self.k_proj(x) #(B,S,head_size)
        value = self.v_proj(x) #(B,S,head_size)
        
        weight = query @ key.transpose(-2,-1) #(B,S,S)
        
        weight = weight.masked_fill(self.attention_mask[:,:seq_len,:seq_len]==0,-float('inf')) / math.sqrt(self.head_size)
        
        weight = F.softmax(weight,dim=-1) # (B,S,S)
        weight = self.drop_out(weight)
        out = weight @ value #(B,S,head_size)
        
        return out        
    

# 多头注意力机制 合并多个单头
class MultiHeadAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.heads = nn.ModuleList(
            [
                SingleHeadAttention(config)
                for _ in range(config.n_head)
            ]
        )
        self.proj = nn.Linear(config.n_embd,config.n_embd)
        self.dropout = nn.Dropout(config.dropout)



    def forward(self,x):
        output = torch.cat(
            [h(x) for h in self.heads],
            dim=-1
        )
        output = self.proj(output)
        output = self.dropout(output)

        return output  

In [7]:
#Group query Attetnion 
import math
# MQA是 GQA的特殊形式，指所有query 共用一个K，V

class GroupQueryAttention(nn.Module):
    def __init__(self,hidden_dim,nums_head,nums_key_value_head,attention_dropout,block_size):
        super().__init__()
        assert hidden_dim % nums_head == 0
        assert nums_head % nums_key_value_head == 0 #N 个 query head 为一组
        
        self.hidden_dim = hidden_dim
        self.nums_head = nums_head
        self.nums_key_value_head = nums_key_value_head
        self.attention_dropout = attention_dropout
        self.head_dim = hidden_dim // nums_head
        
        self.q = nn.Linear(hidden_dim,self.nums_head * self.head_dim) # out feature_size (nums_head * head_dim)
        
        # k v out shape (nums_key_value_head * head_dim)
        self.k = nn.Linear(hidden_dim,self.nums_key_value_head * self.head_dim)
        self.v = nn.Linear(hidden_dim,self.nums_key_value_head * self.head_dim)
        
        self.o = nn.Linear(hidden_dim,hidden_dim)
        
        self.dropout = nn.Dropout(attention_dropout)
        
        self.register_buffer(
            'attention_mask',
            torch.tril(
                torch.ones(block_size,block_size)
            )
        )
        
    def forward(self,x):
        batch_size, seq_len, hidden_dim = x.size()
        query = self.q(x) #(B,S, self.nums_head * self.head_dim)
        key = self.k(x)  #(B,S, self.nums_key_value_head * self.head_dim)
        value = self.v(x) #(B,S, self.nums_key_value_head * self.head_dim)
        
        # 目标 attention weight 是 （B , nums_head , seq , seq ）
        
        query = query.view(batch_size,seq_len,self.nums_head,self.head_dim)
        key = key.view(batch_size,seq_len,self.nums_key_value_head,self.head_dim)
        value = value.view(batch_size,seq_len,self.nums_key_value_head,self.head_dim)
        
        query = query.transpose(1,2) #(B,nums_head, s , head_dim)
        key = key.transpose(1,2) # (B , nums_key_value_head, s ,head_dim)
        value = value.transpose(1,2) #(B , nums_key_value_head, s ,head_dim)
        
        #repeat key and value
        key = key.repeat_interleave(self.nums_head//self.nums_key_value_head,dim=1)
        value = value.repeat_interleave(self.nums_head//self.nums_key_value_head,dim=1)
        
        attention_weight = query @ key.transpose(2,3) # (B,nums_head,s ,s)
        mask = self.attention_mask.unsqueeze(0).unsqueeze(0)
        attention_weight = attention_weight.masked_fill(mask[:,:,:seq_len,:seq_len]==0,-float('inf'))/math.sqrt(self.head_dim)
        
        atttention_score = F.softmax(attention_weight,dim=-1)
        
        output = atttention_score @ value  # (B, nums_head, s, head_dim)
        output = output.transpose(1, 2).contiguous()
        final_output = self.o(output.view(batch_size,seq_len, -1))
        
        return final_output



x = torch.rand(3, 2, 128)
net = GroupQueryAttention(128, 8, 4, 0.1, 2)
net(x).shape

torch.Size([3, 2, 128])

![jupyter](./mla.png)

![jupyter](./mla_model.png)

![jupyter](./deepseek-v3-model-architecture.png)

In [8]:
# MLA
#首先实现前置函数 RMS_Norm + Rope
class DeepseekV2RMSNorm(nn.Module):
    def __init__(self,hidden_size,eps = 1e-6):
        super().__init__()
        self.hidden_size = hidden_size
        
        self.weight = nn.Parameter(torch.ones(hidden_size))
        self.variance_epsilon = eps
    
    def forward(self,hidden_states):
        input_dtype = hidden_states.dtype
        hidden_states = hidden_states.to(torch.float32)
        variance = hidden_states.pow(2).mean(-1,keepdim=True)
        hidden_states = hidden_states* torch.rsqrt(variance+self.variance_epsilon)
        
        return self.weight * hidden_states.to(input_dtype)


class DeepseekV2RotaryEmbedding(nn.Module):
    def __init__(self, dim, max_position_embeddings=2048, base=10000, device=None):
        super().__init__()

        self.dim = dim
        self.max_position_embeddings = max_position_embeddings
        self.base = base
        inv_freq = 1.0 / (
            self.base ** (torch.arange(0, self.dim, 2).float().to(device) / self.dim)
        )
        self.register_buffer("inv_freq", inv_freq, persistent=False)
        # 较小索引位置对应较低频率
        # 较大的索引位置有较高的频率
        
        # Build here to make `torch.jit.trace` work.
        self._set_cos_sin_cache(
            seq_len=max_position_embeddings,
            device=self.inv_freq.device,
            dtype=torch.get_default_dtype(),
        )
        self.max_seq_len_cached = None

    def _set_cos_sin_cache(self, seq_len, device, dtype):
        self.max_seq_len_cached = seq_len
        t = torch.arange(
            self.max_seq_len_cached, device=device, dtype=self.inv_freq.dtype
        )

        freqs = torch.outer(t, self.inv_freq.to(t.device))
        # Different from paper, but it uses a different permutation in order to obtain the same calculation
        emb = torch.cat((freqs, freqs), dim=-1)
        self.register_buffer("cos_cached", emb.cos().to(dtype), persistent=False)
        self.register_buffer("sin_cached", emb.sin().to(dtype), persistent=False)

    def forward(self, x, seq_len=None):
        # x: [bs, num_attention_heads, seq_len, head_size]
        if self.max_seq_len_cached is None or seq_len > self.max_seq_len_cached:
            self._set_cos_sin_cache(seq_len=seq_len, device=x.device, dtype=x.dtype)

        return (
            self.cos_cached[:seq_len].to(dtype=x.dtype),
            self.sin_cached[:seq_len].to(dtype=x.dtype),
        )

# Copied from transformers.models.llama.modeling_llama.rotate_half
def rotate_half(x):
    """Rotates half the hidden dims of the input."""
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2 :]
    return torch.cat((-x2, x1), dim=-1)

# Copied from transformers.models.llama.modeling_llama.apply_rotary_pos_emb
def apply_rotary_pos_emb(q, k, cos, sin, position_ids, unsqueeze_dim=1):
    cos = cos[position_ids].unsqueeze(unsqueeze_dim)
    sin = sin[position_ids].unsqueeze(unsqueeze_dim)

    b, h, s, d = q.shape
    q = q.view(b, h, s, d // 2, 2).transpose(4, 3).reshape(b, h, s, d)

    b, h, s, d = k.shape
    k = k.view(b, h, s, d // 2, 2).transpose(4, 3).reshape(b, h, s, d)

    q_embed = (q * cos) + (rotate_half(q) * sin)
    k_embed = (k * cos) + (rotate_half(k) * sin)
    return q_embed, k_embed


#####实现MLA
from dataclasses import dataclass

@dataclass
class DeepseekConfig:
    hidden_size: int
    num_heads: int
    max_position_embeddings: int
    rope_theta: float
    attention_dropout: float

    q_lora_rank: int
    qk_rope_head_dim: int
    kv_lora_rank: int
    v_head_dim: int
    qk_nope_head_dim: int
    attention_bias: bool


class MLA(nn.Module):
    def __init__(self,config,):
        super().__init__()
        #MHA 部分
        self.attention_dropput = config.attention_dropout
        self.hidden_size = config.hidden_size
        self.num_heads = config.num_heads
        self.v_head_dim = config.v_head_dim
        
        self.out_proj = nn.Linear(self.num_heads* self.v_head_dim, self.hidden_size,bias=False)
        
        
        #MLA压缩部分
        #down 压缩
        
        ### q是在升维的时候获rope部分，k是在降维的时候获得rope部分
        self.qk_nope_head_dim = config.qk_nope_head_dim
        self.qk_rope_head_dim = config.qk_rope_head_dim
        
        self.q_lora_rank = config.q_lora_rank
        self.kv_lora_rank = config.kv_lora_rank
        
        self.q_down_proj = nn.Linear(self.hidden_size, self.q_lora_rank,bias = config.attention_bias)
        self.q_down_norm = DeepseekV2RMSNorm(self.q_lora_rank)
        
        
        self.kv_down_proj = nn.Linear(self.hidden_size, self.kv_lora_rank + config.qk_rope_head_dim ,bias =config.attention_bias)
        self.kv_down_norm = DeepseekV2RMSNorm(self.kv_lora_rank)
        #up 升维
        self.q_head_dim = config.qk_rope_head_dim + config.qk_nope_head_dim
        self.q_up_proj = nn.Linear(
            self.q_lora_rank,
            self.nums_heads * self.q_head_dim,bias = config.attention_bias
        )
        
        self.kv_up_proj = nn.Linear(self.kv_lora_rank,self.num_heads *(self.q_head_dim - config.qk_nope_head_dim + self.v_head_dim),bias = config.attention_bias)
        
        
        ##rope 部分
        self.rotary_emb = DeepseekV2RotaryEmbedding(
            config.qk_rope_head_dim,
            config.max_position_embeddings,
            config.rope_theta
        )
        
    
    def forward(self,hidden_states,position_ids,attention_mask=None):
        bsz,q_len,_ = hidden_states.size()
        
        # 1.compression
        q = self.q_down_proj(hidden_states)
        q = self.q_down_norm(q)
        q = self.q_up_proj(q) # (B,S,nums_head* self.q_head_dim)
        
        q = q.view(bsz,q_len,self.num_heads,self.q_head_dim).transpose(1,2)
        #(B,num_head,S,self.q_head_dim)
        
        q_nope,q_rope = torch.split(
            q,
            [self.qk_nope_head_dim,self.qk_rope_head_dim],
            dim=-1
        )
        
        
        ##kv
        c_kv = self.kv_down_proj(hidden_states)
        
        c_kv, k_rope = torch.split(
            c_kv,
            [self.kv_lora_rank,self.qk_rope_head_dim],
            dim = -1
        )
        
        k_rope = k_rope.view(
            bsz,q_len,1,self.qk_rope_head_dim
        ).transpose(1,2) #brodcast 
        # (b,1,seq_len.qk_rope_head_dim)
        
        kv = (self.kv_up_proj(
            self.kv_down_norm(
                c_kv
            )
        ).view(bsz,q_len,self.num_heads,self.qk_nope_head_dim + self.v_head_dim ).transpose(1,2))
        
        
        k_nope,value_states = torch.split(
            kv,
            [self.qk_nope_head_dim,self.v_head_dim],
            dim=-1
        )
        
        # apply rope
        
        kv_seq_len = value_states.shape[-2]
        cos , sin = self.rotary_emb(
            value_states,seq_len=kv_seq_len
        )
        
        q_rope, k_rope = apply_rotary_pos_emb(q_rope,k_rope,cos,sin,position_ids)
        
        ###MHA
        # 最终 Q, k, V 的 Shape 都希望是 (batch_size, num_heads, seq_len, head_dim)
        # 其中 q / k 的 head_dim = self.qk_nope_head_dim + self.qk_rope_head_dim
        # v 的 head_dim = self.v_head_dim
        query_states = torch.empty(
            bsz, self.num_heads, q_len, self.q_head_dim, 
            device=k_rope.device
        )
        query_states[:, :, :, :self.qk_nope_head_dim] = q_nope
        query_states[:, :, :, self.qk_nope_head_dim:] = q_rope

        
        key_states = torch.empty(
            bsz,self.num_heads,q_len,self.q_head_dim,device=k_rope.device
        )
        key_states[:, :, :, :self.qk_nope_head_dim] = k_nope
        key_states[:, :, :, self.qk_nope_head_dim:] = k_rope
        
        #attetion_weight
        atten_weight = query_states @ key_states.transpose(2,3)   # (B,num_head,S,q_head_dim) * (B,num_head,q_head_dim,S) -> (B,num_head,S,S)
        atten_weight = atten_weight/math.sqrt(self.q_head_dim)  #(B,num_head,S,S)
        
        if attention_mask is not None:
            attn_weight = torch.masked_fill(
                atten_weight,
                attention_mask == 0,
                float("-inf"),
            )
        
        atten_weight = F.softmax(
            atten_weight, dim=-1, dtype=torch.float32).to(query_states.dtype)

        atten_weight = F.dropout(
            atten_weight, p=self.attention_dropout, training=self.training)
        
        attn_out = atten_weight @ value_states #  (B,num_head,S,S) * (B,nums_head,S,v_head_dim) -> (B,nums_head,S,v_head_dim)
        attn_out = attn_out.transpose(1,2).reshape(bsz,q_len,-1) #(B,S, nums_head * v_head_dim)
        attn_out = self.out_proj(attn_out)
        
        return attn_out,attn_weight
        